# Transformer 第二阶段：从 Cross-Attention 到 Encoder–Decoder

[返回项目首页](https://github.com/Maple-Ze/Transformer-to-VLA) · [上一章：01 · Attention Fundamentals](https://github.com/Maple-Ze/Transformer-to-VLA/blob/main/01_Self_Attention_and_MiniGPT/01_Self_Attention_and_MiniGPT.ipynb)

**Google Colab · PyTorch · 从零实现 · 中文教学版**

本阶段在已掌握的 Self-Attention、Causal Mask、Mini-GPT 基础上继续，不重复第一阶段笔记。采用「概念 → 小实验 → 完整模块 → 序列转换训练 → 生成」的顺序。

**你将学会：** Cross-Attention 的 Q/K/V 来自哪里；为什么原始 Decoder 有三个子层；Post-LN 与 Pre-LN 的结构差异；Teacher Forcing 为什么要右移目标；推理时如何复用 Encoder 的输出。

> 建议：Colab「运行时 → 更改运行时类型」可选择 GPU；这里的小模型在 CPU 也能运行。每个代码单元按顺序执行。

## 0. 环境和常用记号

- `B`：批量；`S`：源序列长度（Encoder）；`T`：目标序列长度（Decoder）；`C`：隐藏维度；`H`：头数；`D=C/H`。
- 与 Self-Attention 的 `[B,T,T]` 不同，Cross-Attention Score 的形状是 `[B,H,T,S]`。

In [ ]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

## 1. 单头 Cross-Attention：最核心的新知识

对于 Encoder 输出 `memory` 与 Decoder 当前表示 `query_x`：

$$
Q=X_{dec}W_Q,\quad K=X_{enc}W_K,\quad V=X_{enc}W_V
$$
$$
A=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right),\qquad Y=AV.
$$

**思考：为什么输出的 Token 数是 `T` 而非 `S`？** 每一行 Query 都会得到一个加权的 Value 结果。

In [ ]:
B, S, T, C = 2, 6, 4, 32
memory = torch.randn(B, S, C, device=device)  # Encoder
query_x = torch.randn(B, T, C, device=device) # Decoder

Wq = nn.Linear(C, C, bias=False).to(device)
Wk = nn.Linear(C, C, bias=False).to(device)
Wv = nn.Linear(C, C, bias=False).to(device)
Q, K, V = Wq(query_x), Wk(memory), Wv(memory)
scores = Q @ K.transpose(-2, -1) / math.sqrt(C)
weights = scores.softmax(dim=-1)
context = weights @ V
for name, tensor in [('Q',Q),('K',K),('V',V),('scores',scores),('weights',weights),('context',context)]:
    print(f'{name:8s}', tuple(tensor.shape))
assert scores.shape == (B,T,S) and context.shape == (B,T,C)
assert torch.allclose(weights.sum(-1), torch.ones(B,T,device=device))

### 1.1 看注意力热力图

每一行表示一个 Decoder Query 对全部 Encoder Token 的权重。参数尚未训练，**颜色深不代表有语义的对齐关系**。

In [ ]:
plt.figure(figsize=(6,3.5))
plt.imshow(weights[0].detach().cpu(), aspect='auto', cmap='viridis')
plt.colorbar(label='Attention weight')
plt.xlabel('Encoder key position (S)')
plt.ylabel('Decoder query position (T)')
plt.title('Untrained cross-attention | batch 0')
plt.show()

## 2. Multi-Head Cross-Attention：每个 Head 都能访问整个 Encoder 序列

每个 Head 分别生成 `Q_h:[B,H,T,D]`、`K_h,V_h:[B,H,S,D]`，然后 `Attn_h:[B,H,T,S]`。**注意：Q 的长度 T 与 K/V 的长度 S 可以不同。**

我们同时支持 Self-Attention：当 `query_x == memory_x`（同源张量）时，即为 Self-Attention；用同一个实现避免复制代码。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, L, C = x.shape
        return x.reshape(B, L, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, query_x, memory_x, causal=False, return_attention=False):
        B, T, C = query_x.shape
        S = memory_x.size(1)
        q = self.split_heads(self.q_proj(query_x)) # [B,H,T,D]
        k = self.split_heads(self.k_proj(memory_x)) # [B,H,S,D]
        v = self.split_heads(self.v_proj(memory_x)) # [B,H,S,D]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if causal:
            # Only valid for aligned decoder self-attention positions
            assert T == S, 'This causal mask assumes query/key positions align.'
            allow = torch.ones(T, S, device=scores.device, dtype=torch.bool).tril()
            scores = scores.masked_fill(~allow, float('-inf'))
        attn = scores.softmax(dim=-1)
        z = self.dropout(attn) @ v # [B,H,T,D]
        z = z.transpose(1, 2).contiguous().reshape(B, T, C)
        out = self.out_proj(z)
        return (out, attn) if return_attention else out

cross = MultiHeadAttention(32, 4).to(device)
out, a = cross(query_x, memory, return_attention=True)
print('Cross output:', tuple(out.shape), ' | Heads attention:', tuple(a.shape))
assert a.shape == (2,4,4,6)
self_out, self_a = cross(query_x, query_x, causal=True, return_attention=True)
print('Causal self attention:', tuple(self_a.shape))
assert torch.all(self_a.triu(diagonal=1) == 0)

## 3. 原始 Transformer Encoder Block（Post-LN）

原论文使用 `LayerNorm(x + Sublayer(x))`，Encoder 一层只有：

1. **双向 Self-Attention**：可查看任意源 Token。
2. **Feed Forward Network**：逐 Token 对 Channel 做映射。

$$
U=\mathrm{LN}(X+\mathrm{Dropout}(\mathrm{SelfAttn}(X))),\qquad
Y=\mathrm{LN}(U+\mathrm{Dropout}(\mathrm{FFN}(U)))
$$

为了教学，FFN 的中间维度取 `4*d_model`，原始论文基本配置 `d_model=512,d_ff=2048`。

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.ReLU(),
                                 nn.Linear(4*d_model, d_model))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm1(x + self.drop(self.attn(x, x, causal=False)))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x

enc_block = EncoderBlock(32, 4).to(device)
print('Encoder output:', tuple(enc_block(memory).shape))

## 4. 原始 Transformer Decoder Block（Post-LN）

Decoder 相比 Encoder 多一个 **Cross-Attention**，总共有三个子层：

1. Masked Self-Attention：Decoder 不能偷看未来的目标 Token。
2. Cross-Attention：**Q 来自 Decoder，K/V 来自 Encoder**；这里通常不使用因果下三角掩码，因为源序列是已经给出的完整输入。
3. FFN：逐 Token 映射。

三个子层都包含 Residual + LayerNorm（Post-LN）。

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.ReLU(),
                                 nn.Linear(4*d_model, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, return_cross_attention=False):
        x = self.norm1(x + self.drop(self.self_attn(x, x, causal=True)))
        cross_out, cross_w = self.cross_attn(x, memory, return_attention=True)
        x = self.norm2(x + self.drop(cross_out))
        x = self.norm3(x + self.drop(self.ffn(x)))
        return (x, cross_w) if return_cross_attention else x

dec_block = DecoderBlock(32,4).to(device)
dec_out, cross_weights = dec_block(query_x, memory, return_cross_attention=True)
print('Decoder output:', tuple(dec_out.shape), 'Cross weights:', tuple(cross_weights.shape))

## 5. 固定 Sin/Cos Positional Encoding

这次采用 **原始 Transformer 的 Sinusoidal Position Encoding**，而不是第一阶段 MiniGPT 的可学习位置表。

$$
PE(pos,2i)=\sin(pos/10000^{2i/C}),\quad PE(pos,2i+1)=\cos(pos/10000^{2i/C}).
$$

位置编码**不是可训练参数**。Encoder 和 Decoder 的位置从各自的序列起点分别计数。

In [ ]:
class SinusoidalPosition(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        assert d_model % 2 == 0
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0,d_model,2).float() * (-math.log(10000.0)/d_model))
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0)) # [1,max_len,C], not learned

    def forward(self, x):
        return x + self.pe[:,:x.size(1),:]

pe = SinusoidalPosition(32)
plt.figure(figsize=(7,3.4))
plt.imshow(pe.pe[0,:32].numpy(), cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
plt.xlabel('Channel'); plt.ylabel('Position'); plt.title('Fixed sinusoidal position encoding')
plt.colorbar(); plt.show()

## 6. 训练任务：输入一个数字序列，输出它的反转序列

与之前简单的 `+1` 任务不同，这次 Decoder 必须从 Encoder 输入中读取信息：

`source = [2,7,4,9]` → `target = [9,4,7,2,EOS]`

词表使用 `0~9` 表示数字、`BOS=10` 表示开始生成、`EOS=11` 表示结束；所以 `vocab_size=12`。

**Teacher Forcing（教师强制）**：训练时向 Decoder 提供**真实目标序列右移一位**后的输入：

- `decoder_input = [BOS, 9, 4, 7, 2]`
- `decoder_target = [9, 4, 7, 2, EOS]`

注意：Decoder 的 Causal Mask 仍然必要。否则前面位置可能看见后面的目标，从而发生信息泄漏。

In [ ]:
DIGITS = 10
BOS, EOS = 10, 11
VOCAB = 12
SOURCE_LEN = 4

def make_reverse_batch(batch_size=64, source_len=SOURCE_LEN, device=device):
    src = torch.randint(0, DIGITS, (batch_size, source_len), device=device)
    end = torch.full((batch_size, 1), EOS, dtype=torch.long, device=device)
    target = torch.cat([src.flip(dims=[1]), end], dim=1)
    begin = torch.full((batch_size, 1), BOS, dtype=torch.long, device=device)
    decoder_input = torch.cat([begin, target[:, :-1]], dim=1)
    return src, decoder_input, target

src, dec_in, target = make_reverse_batch(2)
print('source:        ', src.cpu().tolist())
print('decoder input: ', dec_in.cpu().tolist())
print('decoder target:', target.cpu().tolist())
assert src.shape==(2,4) and dec_in.shape==(2,5) and target.shape==(2,5)

## 7. 组合完整 Seq2Seq Transformer

为了易于学习，下面只使用 1 个 Encoder Block + 1 个 Decoder Block；它仍然完整体现 Encoder–Decoder 的信息路径。

- Encoder 输入 `[B,S]` → Encoder memory `[B,S,C]`
- Decoder 输入 `[B,T]`，Cross-Attention 读取 memory → `[B,T,C]`
- 输出 Logits `[B,T,Vocab]`，用全部 Decoder 位置计算 Cross Entropy。

教学版假设所有序列长度固定，无 Padding；因此暂未实现 Padding Mask。

In [ ]:
class TinySeq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size=VOCAB, d_model=32, num_heads=4, dropout=0.0):
        super().__init__()
        self.src_embed = nn.Embedding(vocab_size, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPosition(d_model)
        self.encoder = EncoderBlock(d_model, num_heads, dropout)
        self.decoder = DecoderBlock(d_model, num_heads, dropout)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.scale = math.sqrt(d_model)

    def encode(self, src):
        x = self.pos(self.src_embed(src) * self.scale)
        return self.encoder(x)

    def decode(self, decoder_input, memory, return_attention=False):
        x = self.pos(self.tgt_embed(decoder_input) * self.scale)
        return self.decoder(x, memory, return_cross_attention=return_attention)

    def forward(self, src, decoder_input, targets=None, return_attention=False):
        memory = self.encode(src)
        if return_attention:
            hidden, attn = self.decode(decoder_input, memory, return_attention=True)
        else:
            hidden = self.decode(decoder_input, memory)
        logits = self.lm_head(hidden)
        loss = None if targets is None else F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return (logits, loss, attn) if return_attention else (logits, loss)

model = TinySeq2SeqTransformer().to(device)
logits, loss, attn = model(src, dec_in, target, return_attention=True)
print('logits', tuple(logits.shape), '| loss', float(loss.item()), '| cross attn', tuple(attn.shape))
print('trainable parameters:', sum(p.numel() for p in model.parameters()))
assert logits.shape==(2,5,VOCAB) and attn.shape==(2,4,5,4)

## 8. 训练与观察 Loss

这里主要验证 **Encoder–Decoder 确实能学会反转**，不是追求最佳性能。为保证 CPU 也能训练，默认 `300` 步；若效果不够好，可增加到 `600~1000`。

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.0)
steps = 300   # 想更高准确率，可改为 600 或 1000
batch_size = 64
losses = []
model.train()
for step in range(1, steps+1):
    src, dec_in, targets = make_reverse_batch(batch_size)
    logits, loss = model(src, dec_in, targets)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())
    if step==1 or step%50==0:
        print(f'step {step:4d} | loss {loss.item():.4f}')
plt.figure(figsize=(7,3))
plt.plot(losses)
plt.xlabel('Training step'); plt.ylabel('Cross entropy'); plt.title('Seq2Seq training loss')
plt.show()

## 9. 推理：Encoder 先执行一次，Decoder 逐 Token 生成

这一步和第一阶段 MiniGPT 不同：**Encoder 接收源序列并计算一次 memory**；随后 Decoder 每次根据已有目标前缀产生下一个 Token，通过 Cross-Attention 持续读取相同的 memory。

这里仍然使用 Greedy Decoding（argmax），为便于理解不添加 KV Cache。

In [ ]:
@torch.no_grad()
def translate_reverse(model, source_digits, max_new_tokens=8):
    model.eval()
    source = torch.tensor([source_digits], dtype=torch.long, device=device)
    memory = model.encode(source) # only once
    generated = torch.tensor([[BOS]], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        hidden = model.decode(generated, memory)
        logits = model.lm_head(hidden)
        next_id = logits[:,-1,:].argmax(dim=-1,keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
        if (next_id == EOS).all():
            break
    return generated[0].cpu().tolist()

for example in [[2,7,4,9], [8,1,3,5], [0,9,0,3]]:
    print('source:', example,
          '| expected:', list(reversed(example))+[EOS],
          '| generated:', translate_reverse(model, example))

## 10. 看训练后的 Cross-Attention

图像横轴为 Encoder 输入位置，纵轴为 Decoder 输入位置。反转任务中，一些 Head **可能**出现反向对齐趋势，但不能假设所有 Head 必须如此，注意力热力图也不能单独证明模型的语义理解。

In [ ]:
model.eval()
src = torch.tensor([[2,7,4,9]], device=device)
dec_in = torch.tensor([[BOS,9,4,7,2]], device=device)
with torch.no_grad():
    logits, _, attention = model(src, dec_in, return_attention=True)
fig, axes = plt.subplots(1, 4, figsize=(12,2.8))
for h, ax in enumerate(axes):
    ax.imshow(attention[0,h].cpu(), vmin=0, vmax=1, cmap='viridis', aspect='auto')
    ax.set_title(f'Head {h}')
    ax.set_xlabel('Encoder position')
    if h == 0: ax.set_ylabel('Decoder position')
plt.tight_layout()
plt.show()
print('Predictions with teacher forcing:', logits.argmax(-1)[0].cpu().tolist())

## 11. 最后五个检查问题（建议运行完再回答）

1. Cross-Attention 中 Q/K/V 分别来自哪里？为什么 Score 形状为 `[B,H,T,S]`？
2. 原始 Decoder 的 **Masked Self-Attention** 和 **Cross-Attention** 在可见范围上有何区别？
3. 为什么训练 Decoder 输入必须是 `[BOS, target[:-1]]`，而不是直接把完整 target 喂给 Decoder？
4. 为什么推理阶段 Encoder 输出只需计算一次，但 Decoder 要循环运行？
5. 如果改成变长、带 Padding 的输入，还缺少什么 Mask？

### 本 Notebook 的范围与下一步

本阶段完成「Cross-Attention → 原始 Post-LN Encoder–Decoder → Teacher Forcing → Seq2Seq 推理」。之后可学习 **Vision Transformer（ViT）**，再延伸至视觉/语言/动作 Token 如何进入 VLA。

**后续学习总结的范围从第二阶段开始；不再重复前一份 Markdown 已有的第一阶段内容。**

---

[上一章：01 · Attention Fundamentals](https://github.com/Maple-Ze/Transformer-to-VLA/blob/main/01_Self_Attention_and_MiniGPT/01_Self_Attention_and_MiniGPT.ipynb) · [返回项目首页](https://github.com/Maple-Ze/Transformer-to-VLA)